In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

# Load cleaned data
train = pd.read_csv('../data/processed/train_cleaned.csv')
test = pd.read_csv('../data/processed/test_cleaned.csv')

print("Train shape:", train.shape)
print("Test shape:", test.shape)

Train shape: (1458, 80)
Test shape: (1459, 79)


In [5]:
# Total square footage of entire house
train['TotalSF'] = train['TotalBsmtSF'] + train['1stFlrSF'] + train['2ndFlrSF']
test['TotalSF'] = test['TotalBsmtSF'] + test['1stFlrSF'] + test['2ndFlrSF']

# Verify
print("TotalSF min:", train['TotalSF'].min())
print("TotalSF max:", train['TotalSF'].max())
print("TotalSF mean:", train['TotalSF'].mean())

# Check correlation with SalePrice
print("\nCorrelation with SalePrice:", train['TotalSF'].corr(train['SalePrice']))

TotalSF min: 334
TotalSF max: 6872
TotalSF mean: 2557.150205761317

Correlation with SalePrice: 0.825326065733635


In [7]:
# Age of house when sold
train['HouseAge'] = train['YrSold'] - train['YearBuilt']
test['HouseAge'] = test['YrSold'] - test['YearBuilt']

# Years since last remodel
train['RemodAge'] = train['YrSold'] - train['YearRemodAdd']
test['RemodAge'] = test['YrSold'] - test['YearRemodAdd']

# Check correlations
print("HouseAge correlation with SalePrice:", train['HouseAge'].corr(train['SalePrice']))
print("RemodAge correlation with SalePrice:", train['RemodAge'].corr(train['SalePrice']))

HouseAge correlation with SalePrice: -0.5877667506602265
RemodAge correlation with SalePrice: -0.5685285173432842


In [9]:
# Combine all bathrooms into one feature
train['TotalBathrooms'] = (train['FullBath'] + 
                           train['BsmtFullBath'] + 
                           (train['HalfBath'] * 0.5) + 
                           (train['BsmtHalfBath'] * 0.5))

test['TotalBathrooms'] = (test['FullBath'] + 
                          test['BsmtFullBath'] + 
                          (test['HalfBath'] * 0.5) + 
                          (test['BsmtHalfBath'] * 0.5))

print("TotalBathrooms correlation with SalePrice:", train['TotalBathrooms'].corr(train['SalePrice']))
print(train['TotalBathrooms'].value_counts().head())

TotalBathrooms correlation with SalePrice: 0.6766776974808344
TotalBathrooms
2.0    456
2.5    295
1.0    228
3.0    186
3.5    144
Name: count, dtype: int64


In [11]:
# Combine all porch areas into one feature
train['TotalPorch'] = (train['OpenPorchSF'] + 
                       train['EnclosedPorch'] + 
                       train['3SsnPorch'] + 
                       train['ScreenPorch'] +
                       train['WoodDeckSF'])

test['TotalPorch'] = (test['OpenPorchSF'] + 
                      test['EnclosedPorch'] + 
                      test['3SsnPorch'] + 
                      test['ScreenPorch'] +
                      test['WoodDeckSF'])

print("TotalPorch correlation with SalePrice:", train['TotalPorch'].corr(train['SalePrice']))

TotalPorch correlation with SalePrice: 0.3996953100536832


In [13]:
 # Binary features - 1 if exists, 0 if not
train['HasPool'] = (train['PoolArea'] > 0).astype(int)
train['HasGarage'] = (train['GarageArea'] > 0).astype(int)
train['HasFireplace'] = (train['Fireplaces'] > 0).astype(int)
train['HasBasement'] = (train['TotalBsmtSF'] > 0).astype(int)

test['HasPool'] = (test['PoolArea'] > 0).astype(int)
test['HasGarage'] = (test['GarageArea'] > 0).astype(int)
test['HasFireplace'] = (test['Fireplaces'] > 0).astype(int)
test['HasBasement'] = (test['TotalBsmtSF'] > 0).astype(int)

# Check correlations
binary_features = ['HasPool', 'HasGarage', 'HasFireplace', 'HasBasement']
for col in binary_features:
    print(f"{col} correlation with SalePrice: {train[col].corr(train['SalePrice']):.3f}")

HasPool correlation with SalePrice: 0.077
HasGarage correlation with SalePrice: 0.323
HasFireplace correlation with SalePrice: 0.510
HasBasement correlation with SalePrice: 0.200


In [15]:
new_features = ['TotalSF', 'HouseAge', 'RemodAge', 
                'TotalBathrooms', 'TotalPorch',
                'HasPool', 'HasGarage', 'HasFireplace', 'HasBasement']

print("New features summary:")
print(train[new_features].describe())

print("\nNew correlations with SalePrice:")
for col in new_features:
    print(f"{col}: {train[col].corr(train['SalePrice']):.3f}")

New features summary:
           TotalSF     HouseAge     RemodAge  TotalBathrooms   TotalPorch  \
count  1458.000000  1458.000000  1458.000000     1458.000000  1458.000000   
mean   2557.150206    36.598080    22.982167        2.207476   180.810014   
std     774.109803    30.240565    20.636501        0.781341   156.120838   
min     334.000000     0.000000     0.000000        1.000000     0.000000   
25%    2008.500000     8.000000     4.000000        2.000000    45.000000   
50%    2473.000000    35.000000    14.000000        2.000000   164.000000   
75%    3002.250000    54.000000    41.000000        2.500000   265.000000   
max    6872.000000   136.000000    60.000000        6.000000  1027.000000   

           HasPool    HasGarage  HasFireplace  HasBasement  
count  1458.000000  1458.000000   1458.000000  1458.000000  
mean      0.004115     0.944444      0.526749     0.974623  
std       0.064040     0.229140      0.499455     0.157322  
min       0.000000     0.000000      0.0

In [17]:
os.makedirs('../data/processed', exist_ok=True)

train.to_csv('../data/processed/train_featured.csv', index=False)
test.to_csv('../data/processed/test_featured.csv', index=False)

print("Feature engineered data saved!")
print("Train shape:", train.shape)
print("Test shape:", test.shape)

Feature engineered data saved!
Train shape: (1458, 89)
Test shape: (1459, 88)
